# Lab A — Backend Foundation

**Balanced Vibe Coding · Estimated time: 1 hour**

This section follows one backend journey:

```text
GET /health → POST /upload?chat_id=day2-demo → POST /chat
```


> **Vibe Coding rule**
>
> For every tool-assisted task, students must first explain **inputs**, **expected output**, **allowed files/modification boundary**, and **non-goals**. Only then may Claude Code inspect the current implementation and generate the **smallest possible diff**. Whole-file rewrites are not the default.
>
> Students review every hunk with `git diff` and run the real evidence themselves: commands, `/docs`, browser Network, builds, logs, or platform checks. Claude Code saying “done” is never evidence.
>
> Full prompts are hidden references. Open one only after writing your own contract; use it to compare missing constraints, not as the first step.

> **Direct-code exceptions:** The supplied page-aware PDF parser and Grounded OpenRouter service are read, explained, and tested by students; they are not Claude Code file-generation tasks.


## 1.1 Fork, Clone, and Protect the Key

**Starting Point:** [https://github.com/yangzhr1/smartLearn-AI.git](https://github.com/yangzhr1/smartLearn-AI.git)

1. Fork the repository to your GitHub account.
2. Clone **your fork**.
3. Copy `.env.example` to `.env`.
4. Set `OPENROUTER_API_KEY` in `.env` to your own key.
5. Confirm that Git ignores `.env`.

Never put the real key in `.env.example`, Claude Code, screenshots, frontend code, or Git.


In [ ]:
git clone https://github.com/<YOUR_GITHUB_USERNAME>/smartLearn-AI.git
cd smartLearn-AI
git rev-parse --short HEAD
git status
cp .env.example .env                 # PowerShell: Copy-Item .env.example .env
# Edit .env: OPENROUTER_API_KEY=your_own_key
git check-ignore -v .env


## 1.2 Create the Feature Branch

Create the branch directly, then confirm the active name before editing files.


In [ ]:
git switch -c feature/day2-lite
git branch --show-current


## 1.3 Create the Backend Skeleton

This is a deterministic filesystem task, so create only the agreed structure directly:

```text
smartlearn-backend/
├── __init__.py
├── main.py
├── requirements.txt
└── services/
    └── __init__.py
```


In [ ]:
mkdir -p smartlearn-backend/services
touch smartlearn-backend/__init__.py smartlearn-backend/main.py smartlearn-backend/requirements.txt smartlearn-backend/services/__init__.py

printf "
test_files/
" >> .gitignore


## 1.4 Choose and Install Dependencies

#### Create the virtual environment


In [ ]:
# Run in the terminal (not in the Python interactive shell)
# Mac / Linux:
# python3 -m venv venv
#
# Windows:
# python -m venv venv

#### Activate the virtual environment

In [ ]:
# Mac / Linux:
# source venv/bin/activate
#
# Windows (PowerShell):
# .\venv\Scripts\Activate.ps1
#
# Windows (CMD):
# venv\Scripts\activate.bat

Write the six agreed packages directly into `smartlearn-backend/requirements.txt`: FastAPI, Uvicorn, multipart decoding, PDF parsing, the OpenAI-compatible client, and dotenv loading.

In [ ]:
cat > smartlearn-backend/requirements.txt <<'EOF'
fastapi
uvicorn[standard]
python-multipart
pypdf
openai
python-dotenv
EOF

python -m pip install -r smartlearn-backend/requirements.txt


## 1.5 Vibe Code the Smallest FastAPI App

### Student-first task — write the contract before using Claude Code

Before opening the reference, explain these four items in your own words:

1. **Inputs:** What files, state, request data, or existing behavior does this task receive?
2. **Expected output:** What observable behavior or response must exist when the task is complete?
3. **Modification boundary:** Which exact files may change? Which files and responsibilities must remain untouched?
4. **Non-goals:** What must not be added, redesigned, or generalized?

Then give Claude Code **your contract**, ask it to inspect the current files, and request the **smallest possible diff**. Do not request or accept a whole-file rewrite. If the task is read-only, the correct result is **no diff**.

After the tool responds:

- compare the proposed/actual change with your boundary;
- inspect `git diff -- <allowed-paths>` for every edit;
- reject unrelated hunks;
- run the stated command, `/docs`, browser Network check, build, or platform check yourself;
- record expected versus actual evidence before asking for another edit.

<details>
<summary><strong>Hidden reference prompt — minimal FastAPI app</strong> (open only after writing your own contract)</summary>

```text
Inspect smartlearn-backend/main.py and do not edit yet.
Plan the smallest FastAPI app with app = FastAPI(title="SmartLearn Lite API") and GET / returning {"message": "SmartLearn Lite API is running"}.
Do not add health, PDF, upload, LLM, database, authentication, or frontend code.
Name the exact edit and wait for approval.
```

</details>

Approve the one-file edit, inspect `git diff -- smartlearn-backend/main.py`, then start Uvicorn.


In [ ]:
uvicorn smartlearn-backend.main:app --reload


## 1.6 Verify the App in `/docs`

Open `http://127.0.0.1:8000/docs`, execute `GET /`, and confirm status 200 plus the planned JSON response. Students—not Claude Code—run this evidence check.


## 1.7 Vibe Code and Verify `/health`

Define the contract first:

```text
GET /health → status 200 → {"ok": true}
```

### Student-first task — write the contract before using Claude Code

Before opening the reference, explain these four items in your own words:

1. **Inputs:** What files, state, request data, or existing behavior does this task receive?
2. **Expected output:** What observable behavior or response must exist when the task is complete?
3. **Modification boundary:** Which exact files may change? Which files and responsibilities must remain untouched?
4. **Non-goals:** What must not be added, redesigned, or generalized?

Then give Claude Code **your contract**, ask it to inspect the current files, and request the **smallest possible diff**. Do not request or accept a whole-file rewrite. If the task is read-only, the correct result is **no diff**.

After the tool responds:

- compare the proposed/actual change with your boundary;
- inspect `git diff -- <allowed-paths>` for every edit;
- reject unrelated hunks;
- run the stated command, `/docs`, browser Network check, build, or platform check yourself;
- record expected versus actual evidence before asking for another edit.

<details>
<summary><strong>Hidden reference prompt — GET /health</strong> (open only after writing your own contract)</summary>

```text
Inspect smartlearn-backend/main.py and do not edit yet.
Plan one small change that adds GET /health returning exactly {"ok": true}.
Keep GET / unchanged. Do not add PDF, upload, LLM, database, authentication, or frontend code.
Name the exact location you will edit and explain how I will test it in /docs. Wait for approval.
```

</details>

Approve only the scoped `smartlearn-backend/main.py` edit, inspect `git diff`, and locate the decorator, function, and Python boolean `True`. Then verify `/health` in `/docs`. If it fails, give Claude Code the exact status/body or traceback and request diagnosis before another edit.


## 1.8 Add Page-Aware PDF Parsing

Since this step was already practiced on Day 1, please directly copy the implementation below.

Create `smartlearn-backend/services/pdf.py`:

```python
from io import BytesIO

from pypdf import PdfReader

MAX_PAGES = 30


def extract_pages(pdf_bytes: bytes) -> list[dict]:
    reader = PdfReader(BytesIO(pdf_bytes))

    if len(reader.pages) > MAX_PAGES:
        raise ValueError(f"PDF must contain at most {MAX_PAGES} pages")

    return [
        {
            "page": page_number,
            "text": (page.extract_text() or "").strip(),
        }
        for page_number, page in enumerate(reader.pages, start=1)
    ]
```

`pypdf` extracts text; it does not perform OCR. A scanned PDF may therefore contain pages but no readable text.


## 1.9 Vibe Code Temporary State and `POST /upload?chat_id=`

### Student-first task — write the contract before using Claude Code

Before opening the reference, explain these four items in your own words:

1. **Inputs:** What files, state, request data, or existing behavior does this task receive?
2. **Expected output:** What observable behavior or response must exist when the task is complete?
3. **Modification boundary:** Which exact files may change? Which files and responsibilities must remain untouched?
4. **Non-goals:** What must not be added, redesigned, or generalized?

Then give Claude Code **your contract**, ask it to inspect the current files, and request the **smallest possible diff**. Do not request or accept a whole-file rewrite. If the task is read-only, the correct result is **no diff**.

After the tool responds:

- compare the proposed/actual change with your boundary;
- inspect `git diff -- <allowed-paths>` for every edit;
- reject unrelated hunks;
- run the stated command, `/docs`, browser Network check, build, or platform check yourself;
- record expected versus actual evidence before asking for another edit.

<details>
<summary><strong>Hidden reference prompt — temporary state and POST /upload?chat_id=</strong> (open only after writing your own contract)</summary>

```text
Inspect smartlearn-backend/main.py and smartlearn-backend/services/pdf.py. Do not edit yet.

Plan the smallest temporary upload implementation:
- define one module-level documents dictionary
- POST /upload?chat_id= accepts chat_id as a required query parameter and one UploadFile field named file
- reject non-PDF and empty files with HTTP 400
- read the upload into bytes and call extract_pages(pdf_bytes) without saving the file
- reject a PDF with zero readable text using HTTP 422 and explain that OCR is not supported
- store only the page records with documents[chat_id] = pages
- return status, filename, pages, and characters

Do not add local file storage, OCR, RAG, database, authentication, routers, classes, background jobs, or frontend code.
List the exact files and code locations you will change, then wait for approval.
```

</details>

Before approval, check that the plan changes only `smartlearn-backend/main.py`. After implementation:

1. inspect `git diff`;
2. trace `receive → validate → read bytes → parse → store → respond`;
3. confirm that no upload path or cleanup path exists because files are never saved;
4. run the success/failure tests in Section 1.10 yourself.

For a failure, send Claude Code the expected status, actual status/body, and relevant server log. Ask for the smallest diagnosis and repair—not a route rewrite.


## 1.10 Verify `/upload`

Use files in test_files/ for testing. 

Use `/docs` to test only the important cases:

| Input | Expected result |
|---|---|
| Text PDF under 30 pages | `200`; page count and total character count are reasonable |
| Non-PDF | `400` |
| Empty file | `400` |
| Scanned PDF with no text | `422` with an explanation that OCR is not supported |
| PDF over 30 pages | `400` |

Use a known query value such as `/upload?chat_id=day2-demo`, then reuse that same `chat_id` for `/chat`. Do not restart Uvicorn between the two requests, because the temporary `documents` dictionary is cleared on restart.


## 1.11 Add the Grounded OpenRouter Service

Since this step was already practiced on Day 1, please directly copy the implementation below. 

Create `smartlearn-backend/services/llm.py`:

```python
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

SYSTEM_PROMPT = (
    "You answer messages only from the supplied PDF text. "
    "Cite factual claims with [Page X]. "
    "If the answer is not in the PDF, say that the document does not provide enough information. "
    "Never invent a page number."
)


def answer_from_pages(pages: list[dict], message: str) -> str:
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not configured")

    document_text = "\n\n".join(
        f"### [Page {page['page']}]\n{page['text']}"
        for page in pages
        if page["text"]
    )

    client = OpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
    )
    response = client.chat.completions.create(
        model=os.getenv("OPENROUTER_MODEL", "openrouter/free"),
        temperature=0.0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"PDF text:\n{document_text}\n\nmessage: {message}",
            },
        ],
    )
    return response.choices[0].message.content or ""
```

The API key is loaded only when `/chat` calls the service, so `/health` can still work when the key is missing.


## 1.12 Vibe Code `POST /chat`

The supplied `answer_from_pages()` service is ready. Students now use Vibe Coding to connect it to the temporary document store.

Send:

### Student-first task — write the contract before using Claude Code

Before opening the reference, explain these four items in your own words:

1. **Inputs:** What files, state, request data, or existing behavior does this task receive?
2. **Expected output:** What observable behavior or response must exist when the task is complete?
3. **Modification boundary:** Which exact files may change? Which files and responsibilities must remain untouched?
4. **Non-goals:** What must not be added, redesigned, or generalized?

Then give Claude Code **your contract**, ask it to inspect the current files, and request the **smallest possible diff**. Do not request or accept a whole-file rewrite. If the task is read-only, the correct result is **no diff**.

After the tool responds:

- compare the proposed/actual change with your boundary;
- inspect `git diff -- <allowed-paths>` for every edit;
- reject unrelated hunks;
- run the stated command, `/docs`, browser Network check, build, or platform check yourself;
- record expected versus actual evidence before asking for another edit.

<details>
<summary><strong>Hidden reference prompt — POST /chat</strong> (open only after writing your own contract)</summary>

```text
Inspect smartlearn-backend/main.py and smartlearn-backend/services/llm.py. Do not edit yet.

Plan the smallest POST /chat implementation:
- define ChatRequest with chat_id defaulting to "day2-demo" and message limited to 2–2000 characters
- retrieve pages directly from the existing documents dictionary
- return 404 with re-upload guidance when the ID is missing
- call answer_from_pages(pages, message)
- turn an upstream AI failure into a safe HTTP 502
- extract distinct [Page X] numbers from the answer, keep only page numbers that exist in pages, and sort them
- return {"answer": ..., "citations": [...]}

Do not add history, RAG, database, authentication, streaming, retries, or frontend code.
Trace request JSON → stored pages → LLM service → response JSON, list the exact edits, and wait for approval.
```

</details>

After approval, inspect `git diff` and ask Claude Code to explain why a missing ID is 404 while an upstream model failure is 502. Test a fake ID first, then use the real ID returned by `/upload`. If all real IDs are missing, check whether Uvicorn restarted before changing the route.


## 1.13 Verify the Complete Flow

In one Uvicorn session:

1. `GET /health` → `200`, `{"ok": true}`.
2. `POST /upload?chat_id=day2-demo` with a known PDF → confirm reasonable page and character counts.
3. `POST /chat` with `chat_id=day2-demo` and a known message → answer includes the correct `[Page X]`, and `citations` contains only page numbers present in the uploaded pages.
4. Ask something absent → the answer admits insufficient evidence.
5. Use a fake ID → `404`.

Example `/chat` body:

```json
{
  "chat_id": "day2-demo",
  "message": "What is the main conclusion of the document?"
}
```

Test questions:

```text
1. What is the dimension of each attention head used in the Transformer?
Expected Page: Page 5

2. What optimizer and learning-rate schedule are used to train the Transformer?
Expected Page: Page 7

3. What regularization methods are used to train the Transformer?
Expected Page: Page 7

4. What accuracy does the Transformer achieve on the ImageNet image-classification benchmark?
Expected Page: null
```

The important evidence is the real route sequence—not whether the answer merely sounds plausible.


## 1.14 Configure Environment-Driven CORS

### Purpose

CORS is a required boundary for both the local React client and the deployed Vercel client. The backend must read allowed origins from the environment; a hard-coded localhost list does not satisfy this checkpoint.

### Student-first task — write the contract before using Claude Code

Before opening the reference, explain these four items in your own words:

1. **Inputs:** `smartlearn-backend/main.py` and the optional `ALLOWED_ORIGINS` environment variable.
2. **Expected output:** browser requests from every configured exact origin can read backend responses; unlisted origins remain blocked.
3. **Modification boundary:** change only the CORS import/configuration in `smartlearn-backend/main.py`; do not rewrite routes.
4. **Non-goals:** no wildcard origins or methods, credential cookies, proxy, authentication, or deployment-specific URL in source code.

Then give Claude Code **your contract**, ask it to inspect the current file, and request the **smallest possible diff**. Do not request or accept a whole-file rewrite.

<details>
<summary><strong>Hidden reference prompt — environment-driven CORS</strong> (open only after writing your own contract)</summary>

```text
Inspect smartlearn-backend/main.py and do not edit yet.
Plan the smallest CORS-only change in that file:
- import os and CORSMiddleware only if they are not already imported;
- read ALLOWED_ORIGINS from the environment;
- default to http://localhost:5173;
- split comma-separated values, strip whitespace, and discard empty entries;
- pass the resulting allowed_origins variable to CORSMiddleware;
- never hard-code allow_origins=["http://localhost:5173"];
- set allow_credentials=False;
- allow only GET and POST methods and the required request headers;
- do not use wildcard origins or methods and do not change route behavior.

Show the exact insertion locations and explain why a POST can return 200 in
DevTools while browser fetch still rejects the response when the CORS response
header is missing. Wait for student approval before generating the smallest diff.
After approval, edit only smartlearn-backend/main.py and list the evidence the
student—not the tool—must run.
```

</details>

### Review gate — reject a hard-coded localhost list

The implementation must have this behavior:

```python
allowed_origins = [
    origin.strip()
    for origin in os.getenv(
        "ALLOWED_ORIGINS",
        "http://localhost:5173",
    ).split(",")
    if origin.strip()
]

app.add_middleware(
    CORSMiddleware,
    allow_origins=allowed_origins,
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)
```

Equivalent formatting is acceptable, but `allow_origins=["http://localhost:5173"]` and `allow_methods=["*"]` are not.

### Student-run evidence

Review the diff first:

```bash
git diff -- smartlearn-backend/main.py
```

Start the backend with two exact origins:

```bash
cd smartlearn-backend
ALLOWED_ORIGINS="http://localhost:5173,https://smartlearn-lite.example" \
  uvicorn main:app --reload
```

In another terminal, prove the production-shaped origin receives the CORS response header:

```bash
curl -i -X OPTIONS \
  "http://127.0.0.1:8000/upload?chat_id=day2-demo" \
  -H "Origin: https://smartlearn-lite.example" \
  -H "Access-Control-Request-Method: POST"
```

Expected evidence includes:

```text
access-control-allow-origin: https://smartlearn-lite.example
```

Also rerun `/health → /upload → /chat` to prove that middleware configuration did not change route behavior. Record expected versus actual evidence before continuing to Lab B.


## 1.15 Review and Commit

After `/health → /upload → /chat` passes, review every changed file with `git diff`. Confirm `.env`, upload, test PDFs, caches, and keys are absent before staging.


In [ ]:
git status
git diff
git add smartlearn-backend .gitignore
git diff --staged
git commit -m "feat: add PDF upload and cited message answering"


## Final Checkpoint

- [ ] `GET /health` returns `200` and `{"ok": true}` without PDF or AI work.
- [ ] `/upload?chat_id=` stores page records under the caller-provided ID, returns reasonable page/character evidence, and rejects invalid or unreadable input with the required status.
- [ ] `/chat` returns grounded `answer + citations` for known evidence, admits unknown evidence, validates citation pages, and gives upload-first guidance for a missing `chat_id`.
- [ ] CORS reads comma-separated `ALLOWED_ORIGINS` rather than a hard-coded localhost list, and Git contains no secret, uploaded PDF, or generated directory.

If these four statements are true, the Day 2 backend is complete.
